# Geo info for Chapter 40B Applicant data 2021-25

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
!pip install sqlalchemy_mate==2.0.0.0

In [ ]:
!pip install uszipcode

In [ ]:
from uszipcode import SearchEngine
search = SearchEngine()

## Load dataset

In [ ]:
file_path = "/content/drive/MyDrive/Colab Notebooks/CHAPA_Chapter-40B_Application-Data_2021-2025_merged_v0.3.csv"
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip().str.lower()
df.head()

,id_number,submission_date,age,disability,chapa_races,census_races,current_residence_town_city,current_residence_state,current_residence_zip,application_source,...,race_cleaned,race_type,race_american_indian_or_alaska_native,race_asian,race_black_or_african_american,race_hispanic_or_latino,race_middle_eastern_or_north_african,race_native_hawaiian_or_pacific_islander,race_other__prefer_not_to_answer,race_white
0,150733,2022-02-08,65,0,White/Non-Minority,White alone,Somerville,MA,NaN,Unknown,...,['White'],Single Race,0,0,0,0,0,0,0,1
1,439181,2022-02-08,59,0,Hispanic/Latino,Unknown,Westford,MA,NaN,Unknown,...,['Hispanic or Latino'],Single Race,0,0,0,1,0,0,0,0
2,697399,2022-02-08,0,0,Black or African American,Black or African American alone,Malden,MA,NaN,Unknown,...,['Black or African American'],Single Race,0,0,1,0,0,0,0,0
3,191565,2022-02-08,0,0,White/Non-Minority,White alone,Haverhill,MA,NaN,Unknown,...,['White'],Single Race,0,0,0,0,0,0,0,1
4,226436,2022-02-08,60,0,White/Non-Minority,White alone,Nashua,NH,NaN,Unknown,...,['White'],Single Race,0,0,0,0,0,0,0,1


## Filter certain matched addresses to pull geo info

* Select 'Exact', 'Non_Exact', 'Zillow' rows
* Drop 'Unknown' in current residence

In [ ]:
df = df[
    (df['match_type'].isin(['Exact', 'Non_Exact', 'Zillow'])) &
    (df['current_residence_town_city'] != 'Unknown') &
    (df['current_residence_state'] != 'Unknown') &
    (df['age'] != 0)
].reset_index(drop=True)
len(df)

1465

In [ ]:
# property_zip: extract from matched_address

def extract_zip(addy):
    if pd.isna(addy):
        return None
    match = re.search(r",[^,]*?(\d{5})\s*$", addy)  # look after the last comma and extract 5 digits
    if match:
        return match.group(1)
    return None

df.loc[df["property_zip"].isna(), "property_zip"] = df.loc[df["property_zip"].isna(), "matched_address"].apply(extract_zip)

/tmp/ipython-input-1565990801.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['01845' '01845' '01845' ... '02465' '01772' '01453']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df["property_zip"].isna(), "property_zip"] = df.loc[df["property_zip"].isna(), "matched_address"].apply(extract_zip)


# Get Longitude, Latitude from uszipecode using property_zip

In [ ]:
# property_long, property_lat
property_lat = [0 for i in range(len(df))]
property_lon = [0 for i in range(len(df))]

town_missing = []

for i in range(len(df)):

  zip_code = df['property_zip'][i]

  results = search.by_zipcode(zip_code)

  if results:

    latitude = results.lat
    longitude = results.lng

    property_lat[i] = latitude
    property_lon[i] = longitude

    print(f"--- {i}, {zip_code}, {latitude}, {longitude} ---")

  else:
    print(f"'{i}, {latitude}, {longitude}' could not be find")
    town_missing.append(i)

print(town_missing)

df['property_lat'] = property_lat
df['property_lon'] = property_lon

--- 0, 01845, 42.7, -71.11 ---
--- 1, 01845, 42.7, -71.11 ---
--- 2, 01845, 42.7, -71.11 ---
--- 3, 01730, 42.48, -71.26 ---
--- 4, 01730, 42.48, -71.26 ---
--- 5, 01730, 42.48, -71.26 ---
--- 6, 01730, 42.48, -71.26 ---
--- 7, 02718, 41.87, -71.01 ---
--- 8, 01862, 42.58, -71.3 ---
--- 9, 01862, 42.58, -71.3 ---
--- 10, 01862, 42.58, -71.3 ---
--- 11, 01862, 42.58, -71.3 ---
--- 12, 01862, 42.58, -71.3 ---
--- 13, 01862, 42.58, -71.3 ---
--- 14, 01862, 42.58, -71.3 ---
--- 15, 01862, 42.58, -71.3 ---
--- 16, 01862, 42.58, -71.3 ---
--- 17, 01862, 42.58, -71.3 ---
--- 18, 01862, 42.58, -71.3 ---
--- 19, 01862, 42.58, -71.3 ---
--- 20, 01938, 42.67, -70.83 ---
--- 21, 01938, 42.67, -70.83 ---
--- 22, 01845, 42.7, -71.11 ---
--- 23, 01845, 42.7, -71.11 ---
--- 24, 01845, 42.7, -71.11 ---
--- 25, 01845, 42.7, -71.11 ---
--- 26, 01845, 42.7, -71.11 ---
--- 27, 01845, 42.7, -71.11 ---
--- 28, 01845, 42.7, -71.11 ---
--- 29, 01845, 42.7, -71.11 ---
--- 30, 01845, 42.7, -71.11 ---
--- 31, 018

# Get Zip, Longitude, Latitude from uszipcode using town/city name and state
- get zipcode with most population when there are multiple

In [ ]:
# current_residence_zip, current_res_long, current_res_lat

current_lat = [0 for i in range(len(df))]
current_lon = [0 for i in range(len(df))]

town_drop = [362, 510, 580, 704, 705, 706, 958, 1189, 1326, 1400, 1453]
town_missing = []

for i in range(len(df)):
  if i in town_drop:
    continue

  town_name = df['current_residence_town_city'][i]
  state_abbr = df['current_residence_state'][i]

  results = search.by_city_and_state(city=town_name, state=state_abbr, returns=0)

  if results:
    sorted_zips = sorted(
        results,
        key = lambda x : x.population if x.population is not None else -1,
        reverse = True
    )

    selected_zip = sorted_zips[0]

    zip_code = selected_zip.zipcode
    latitude = selected_zip.lat
    longitude = selected_zip.lng

    df.loc[i, 'current_residence_zip'] = zip_code
    current_lat[i] = latitude
    current_lon[i] = longitude

    print(f"--- {i}. {town_name}, {state_abbr} ({len(results)})---")
    print(f"ZIP: {zip_code} | Lat/Lon: {latitude}, {longitude}")

  else:
    print(f"'{i}, {town_name}, {state_abbr}' could not be find")
    town_missing.append(i)

print(town_missing)

df['current_lat'] = current_lat
df['current_lon'] = current_lon

/tmp/ipython-input-3779638770.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '02145' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[i, 'current_residence_zip'] = zip_code


--- 0. Somerville, MA (3)---
ZIP: 02145 | Lat/Lon: 42.39, -71.1
--- 1. Westford, MA (1)---
ZIP: 01886 | Lat/Lon: 42.58, -71.43
--- 2. Nashua, NH (4)---
ZIP: 03060 | Lat/Lon: 42.74, -71.46
--- 3. Waltham, MA (3)---
ZIP: 02453 | Lat/Lon: 42.37, -71.24
--- 4. Woburn, MA (1)---
ZIP: 01801 | Lat/Lon: 42.48, -71.15
--- 5. Burlington, MA (1)---
ZIP: 01803 | Lat/Lon: 42.5, -71.2
--- 6. Chelmsford, MA (1)---
ZIP: 01824 | Lat/Lon: 42.59, -71.36
--- 7. Taunton, MA (1)---
ZIP: 02780 | Lat/Lon: 41.9, -71.09
--- 8. Burlington, MA (1)---
ZIP: 01803 | Lat/Lon: 42.5, -71.2
--- 9. Woburn, MA (1)---
ZIP: 01801 | Lat/Lon: 42.48, -71.15
--- 10. Waltham, MA (3)---
ZIP: 02453 | Lat/Lon: 42.37, -71.24
--- 11. Beverly, MA (1)---
ZIP: 01915 | Lat/Lon: 42.57, -70.87
--- 12. Burlington, MA (1)---
ZIP: 01803 | Lat/Lon: 42.5, -71.2
--- 13. Lawrence, MA (3)---
ZIP: 01841 | Lat/Lon: 42.71, -71.16
--- 14. Woburn, MA (1)---
ZIP: 01801 | Lat/Lon: 42.48, -71.15
--- 15. Reading, MA (1)---
ZIP: 01867 | Lat/Lon: 42.53, -71.

- handling missing town/city name by manually

In [ ]:
need_handle = town_drop + town_missing
print(need_handle)

for i in range(len(df)):
  if i in need_handle:
    print(df['current_residence_town_city'][i], df['current_residence_state'][i])

# Oak Bluffs MA: 02557 / 41.44 / -70.56
# Pelham MA: 01002 / 42.43 / -72.47
# Nashua MA: 03060 / 42.74 / -71.45
# Easton MA: 02356 / 42.06 / -71.11
# East Providence MA: 02914 / 41.81 / -71.38
# Woodstock MA: 06281 / 41.89 / -71.94
# Woonsocket MA: 02895 / 42.00 / -71.50
# Menomonie WI: 54751 / 44.88 / -91.96
# Hooksett NH: 03106 / 43.09 / -71.46
# Bristol MA: 02780 / 41.63 / -70.92

[362, 510, 580, 704, 705, 706, 958, 1189, 1326, 1400, 1453, 265, 336, 343, 602, 608, 737, 746, 938, 1230, 1345]
Oak Bluffs MA
Oak Bluffs MA
Oak Bluffs MA
Pelham MA
Nashua MA
Nashua MA
Easton MA
East Providence MA
Woodstock MA
Woodstock MA
Woodstock MA
Oak Bluffs MA
Easton MA
Oak Bluffs MA
Woonsocket MA
Menomonie MA
Hooksett MA
Nashua MA
Oak Bluffs MA
Nashua MA
Bristol MA


In [ ]:
for i in range(len(df)):
  if df['current_residence_town_city'][i] == 'Oak Bluffs':
    df.loc[i, 'current_residence_zip'] = '02557'
    df.loc[i, 'current_lat'] = 41.44
    df.loc[i, 'current_lon'] = -70.56
  if df['current_residence_town_city'][i] == 'Pelham':
    df.loc[i, 'current_residence_zip'] = '01002'
    df.loc[i, 'current_lat'] = 42.43
    df.loc[i, 'current_lon'] = -72.47
  elif df['current_residence_town_city'][i] == 'Nashua':
    df.loc[i, 'current_residence_zip'] = '03060'
    df.loc[i, 'current_lat'] = 42.74
    df.loc[i, 'current_lon'] = -71.45
  elif df['current_residence_town_city'][i] == 'Easton':
    df.loc[i, 'current_residence_zip'] = '02356'
    df.loc[i, 'current_lat'] = 42.06
    df.loc[i, 'current_lon'] = -71.11
  elif df['current_residence_town_city'][i] == 'East Providence':
    df.loc[i, 'current_residence_zip'] = '02914'
    df.loc[i, 'current_lat'] = 41.81
    df.loc[i, 'current_lon'] = -71.38
  elif df['current_residence_town_city'][i] == 'Woodstock':
    df.loc[i, 'current_residence_zip'] = '06281'
    df.loc[i, 'current_lat'] = 41.89
    df.loc[i, 'current_lon'] = -71.94
  elif df['current_residence_town_city'][i] ==  'Woonsocket':
    df.loc[i, 'current_residence_zip'] = '02895'
    df.loc[i, 'current_lat'] = 42.0
    df.loc[i, 'current_lon'] = -71.5
  elif df['current_residence_town_city'][i] == 'Menomonie':
    df.loc[i, 'current_residence_state'] = 'WI'
    df.loc[i, 'current_residence_zip'] = '54751'
    df.loc[i, 'current_lat'] = 44.88
    df.loc[i, 'current_lon'] = -91.96
  elif df['current_residence_town_city'][i] == 'Hooksett':
    df.loc[i, 'current_residence_state'] = 'NH'
    df.loc[i, 'current_residence_zip'] = '03106'
    df.loc[i, 'current_lat'] = 43.09
    df.loc[i, 'current_lon'] = -71.46
  elif df['current_residence_town_city'][i] == 'Bristol':
    df.loc[i, 'current_residence_zip'] = '02780'
    df.loc[i, 'current_lat'] = 41.63
    df.loc[i, 'current_lon'] = -70.92

# Calculate the distance between property and current residence using Longitude and Latitude

In [ ]:
# distance
from geopy.distance import geodesic

distance_km = [0 for i in range(len(df))]
distance_mi = [0 for i in range(len(df))]

for i in range(len(df)):
  coords_1 = (df['property_lat'][i], df['property_lon'][i])
  coords_2 = (df['current_lat'][i], df['current_lon'][i])

  dist_km = geodesic(coords_1, coords_2).km
  distance_km[i] = dist_km

  dist_miles = geodesic(coords_1, coords_2).miles
  distance_mi[i] = dist_miles

df['distance_km'] = distance_km
df['distance_mi'] = distance_mi

In [ ]:
print(df['distance_km'].describe())
print(df['distance_mi'].describe())

count    1465.000000
mean       30.908066
std       121.956919
min         0.000000
25%         8.924493
50%        17.894311
75%        32.471481
max      2155.690737
Name: distance_km, dtype: float64
count    1465.000000
mean       19.205382
std        75.780516
min         0.000000
25%         5.545423
50%        11.119010
75%        20.176843
max      1339.484124
Name: distance_mi, dtype: float64


# Making dataframe that includes every counties in MA and each towns it contains

In [ ]:
# current_residence_county
df_county = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/county_town - Sheet1.csv')
print(df_county)

        county                                               town
0   Barnstable  Barnstable Town city, Bourne town, Brewster to...
1        Essex  Amesbury town, Beverly city, Danvers town, Ess...
2    Middlesex  Acton town, Arlington town, Ashby town, Ashlan...
3      Norfolk  Wilmington town, Winchester town, Woburn city,...
4     Plymouth  Carver town, Duxbury town, Hanover town, Hingh...
5      Suffolk  Boston city, Chelsea city, Revere city, Winthr...
6    Berkshire  Alford town, Becket town, Clarksburg town, Egr...
7      Bristol  Easton town, Raynham town, Acushnet town, Dart...
8     Franklin  Taunton city, Ashfield town, Bernardston town,...
9      Hampden  Agawam city, Blandford town, Brimfield town, C...
10   Hampshire  Amherst town, Belchertown town, Chesterfield t...
11   Worcester  Berlin town, Blackstone town, Bolton town, Har...
12       Dukes  Aquinnah town, Chilmark town, Edgartown town, ...
13   Nantucket  Oak Bluffs town, Tisbury town, West Tisbury to...


In [ ]:
df_clean = df_county.assign(town=df_county['town'].str.split(', ')).explode('town')
df_clean['town'] = df_clean['town'].str.replace(r' (city|City)$', '', regex=True).str.strip()
df_clean['town'] = df_clean['town'].str.replace(r' (Town|town)$', '', regex=True).str.strip()
df_final = df_clean[['county', 'town']].reset_index(drop=True)
print(df_final)

         county          town
0    Barnstable    Barnstable
1    Barnstable        Bourne
2    Barnstable      Brewster
3    Barnstable       Chatham
4    Barnstable        Dennis
..          ...           ...
339       Dukes              
340   Nantucket    Oak Bluffs
341   Nantucket       Tisbury
342   Nantucket  West Tisbury
343   Nantucket     Nantucket

[344 rows x 2 columns]


# Mapping county using town/city name of property/current

In [ ]:
county_map = df_final[['town', 'county']].drop_duplicates()
county_map.columns = ['current_residence_town_city', 'current_county']
df = pd.merge(
    df,
    county_map,
    on='current_residence_town_city',
    how='left'
)

In [ ]:
# property county
county_map = df_final[['town', 'county']].drop_duplicates()
county_map.columns = ['property_town_city', 'property_county']
df = pd.merge(
    df,
    county_map,
    on='property_town_city',
    how='left'
)

# Make binary columns of local applicant


- local: if in the same county

In [ ]:
# local_applicant by same county
for i in range(len(df)):
  if df['property_county'][i] == df['current_county'][i]:
    df.loc[i, 'local_applicant_county'] = 1
  else:
    df.loc[i, 'local_applicant_county'] = 0

- local: if in the same town/city


In [ ]:
# local_applicant by same town/city
for i in range(len(df)):
  if df['property_town_city'][i] == df['current_residence_town_city'][i]:
    df.loc[i, 'local_applicant_town'] = 1
  else:
    df.loc[i, 'local_applicant_town'] = 0

- local: if distance between property and county is zero

In [ ]:
# local_applicant by zero distance
for i in range(len(df)):
  if df['distance_mi'][i] == 0:
    df.loc[i, 'local_applicant_distance'] = 1
  else:
    df.loc[i, 'local_applicant_distance'] = 0

In [ ]:
df

,id_number,submission_date,age,disability,chapa_races,census_races,current_residence_town_city,current_residence_state,current_residence_zip,application_source,...,property_lon,current_lat,current_lon,distance_km,distance_mi,current_county,property_county,local_applicant_county,local_applicant_town,local_applicant_distance
0,150733,2022-02-08,65,0,White/Non-Minority,White alone,Somerville,MA,02145,Unknown,...,-71.11,42.39,-71.10,34.445797,21.403626,Middlesex,Essex,0.0,0.0,0.0
1,439181,2022-02-08,59,0,Hispanic/Latino,Unknown,Westford,MA,01886,Unknown,...,-71.11,42.58,-71.43,29.436218,18.290818,Middlesex,Essex,0.0,0.0,0.0
2,226436,2022-02-08,60,0,White/Non-Minority,White alone,Nashua,NH,03060,Unknown,...,-71.11,42.74,-71.45,28.201723,17.523738,NaN,Essex,0.0,0.0,0.0
3,170029,2021-06-15,35,0,White/Non-Minority,White alone,Waltham,MA,02453,Unknown,...,-71.26,42.37,-71.24,12.329329,7.661090,Middlesex,Middlesex,1.0,0.0,0.0
4,569863,2021-06-15,38,0,Black or African American,Black or African American alone,Woburn,MA,01801,Unknown,...,-71.26,42.48,-71.15,9.044772,5.620161,Norfolk,Middlesex,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1460,995246,2024-05-19,47,0,White,White alone,Framingham,MA,01702,Friend/Word of Mouth,...,-71.51,42.28,-71.44,6.185841,3.843703,Middlesex,Worcester,0.0,0.0,0.0
1461,997147,2024-05-21,35,0,Choose not to answer,Missing,Marlborough,MA,01752,MyMassHome,...,-71.51,42.34,-71.54,5.085050,3.159703,Middlesex,Worcester,0.0,0.0,0.0
1462,997147,2024-06-21,35,0,Choose not to answer,Missing,Marlborough,MA,01752,MyMassHome,...,-71.22,42.34,-71.54,26.391924,16.399181,Middlesex,Middlesex,1.0,0.0,0.0
1463,998583,2024-05-09,25,0,"Black or African American, Hispanic or Latino",Black or African American,Bedford,MA,01730,MyMassHome,...,-71.51,42.48,-71.26,28.697616,17.831872,Middlesex,Worcester,0.0,0.0,0.0


In [ ]:
df.to_csv("/content/drive/MyDrive/Colab Notebooks/40B_zip_long_lat.csv", index=False)

# Summary

1. Geocoding data

  The final preprocessed data was used to complete the geocoding data, preparing the answer for Question 1. The Latitude and Longitude values for each applicant were obtained using the external uszipcode module with the zip code, town/city, and state data.

  The distance between the two locations was calculated in miles using the Longitude and Latitude values for both the property and current locations. This established a basis for classifying an applicant's location change as either local or distant.


2. County mapping

  To move beyond simply classifying local vs. distant based on whether the town/city name is the same or different, an analysis was sought using the broader criterion of county change.

  A PDF file containing all Massachusetts (MA) counties and the town/city names included in each county was found, and a CSV file was manually created from it. This CSV file was used to create a county-town/city dataframe, which was then mapped with our data to extract the county information for each applicant's property and current locations.


3. local vs. distant categorize

  Local and distant were categorized based on a total of three criteria:
- Whether there was a county change.
- Whether there was a town/city change.
- Whether there was a mile distance change.

  Since the number of 'local' applicants varies depending on these three criteria, there is a need to establish a clearer standard. This will be finalized later through a client meeting.


4. Next step

Now that the geocoding data has been established, the focus will shift to analyzing the distribution and specific characteristics of demographic features based on the local vs. distant classification.